# Speed-of-sound autofocus: conventional vs transmit-delay-consistent slowness

This notebook runs the differentiable plane-wave beamformer of the paper on **simulated** data, so
no download is needed. Each simulated medium has a known speed of sound; the scanner's firing delays
are computed with $c_0 = 1540$ m/s, as on a real scanner.

The speed of sound is estimated by the maximum of the coherence factor (CF) with two delay models:

* **conventional**: $p = \sin\theta / c$ (the steering angle stays fixed when $c$ changes)
* **transmit-delay-consistent**: $p = \sin\theta / c_0$ (the slowness set by the firing delays stays fixed)

Runtime: about 5 minutes on a CPU (Colab: *Runtime → Run all*).

In [ ]:
import os, sys
if not os.path.exists("demo/demo.py"):
    !git clone -q https://github.com/sdehury34/sos-autofocus.git
    %cd sos-autofocus
!pip install -q h5py hdf5plugin   # JAX is preinstalled on Colab; elsewhere: pip install -r cubdl_sos/requirements.txt

### Interactive app

Run this cell to open the app below (pick a speed, press **Run**). Skip it for the step-by-step version.

In [ ]:
!pip install -q gradio
sys.path.insert(0, "demo")
import app
app.build().queue().launch(share=True)

In [ ]:
sys.path.insert(0, "demo")
import time, demo

speeds = [1500, 1580, 1620]          # true speeds of sound (m/s); change them if you like
res = {}
for c_true in speeds:
    path = demo.simulate(c_true, "demo/out")
    res[c_true] = {}
    for sl in demo.MODELS:
        t = time.time()
        res[c_true][sl] = demo.autofocus(demo.beamformer(path, sl))
        print(f"true {c_true} m/s | {sl:6s}: estimate {res[c_true][sl][0]:.0f} m/s "
              f"(error {res[c_true][sl][0] - c_true:+.0f}, {time.time() - t:.0f} s)")

In [ ]:
from IPython.display import Image
demo.plot(res, "demo/out/demo.png")
Image("demo/out/demo.png")

**What to look for.** (a) The CF maximum of the conventional model lies between the true speed and
1540 m/s; the consistent model peaks at the true speed. (b) The conventional estimates are pulled
toward $c_0$ (slope below 1); the consistent estimates follow the identity line. On the in vivo calf
data of the paper, the same change decreases the mean absolute error from 27.6 to 12.5 m/s.

### Optional: gradient descent through the beamformer

The beamformer is differentiable in $c$ (JAX forward mode), so the CF maximum can also be found by
gradient descent. This takes a few minutes on a CPU.

In [ ]:
from run_sos_comparison import gradient_descent
c_true = speeds[-1]
c, n, _, _ = gradient_descent(demo.beamformer(demo.simulate(c_true, "demo/out"), "delays"), "cf")
print(f"gradient descent from 1540 m/s: {c:.1f} m/s (true {c_true} m/s, {n} gradient evaluations)")